In [10]:
import pandas as pd
import numpy as np

In [13]:
df = pd.read_csv("california_tourist_places.csv")

In [14]:
import re

# Match against place names, Google place types, addresses, and the query
# that returned each result. A row can receive multiple labels.
CATEGORY_RULES = {
    'restaurant_dining': re.compile(r'\b(restaurant|dining|cafe|coffee shop|food court|meal takeaway|meal delivery)\b'),
    'cooking_experience': re.compile(r'\b(cooking class|cooking|culinary class|baking class|food tour|wine tasting|tasting room|brewery tour)\b'),
    'live_music': re.compile(r'\b(live music|music venue|concert hall|concert|jazz|opera|live performance)\b'),
    'sightseeing': re.compile(r'\b(tourist attraction|landmark|scenic spot|observation deck|lookout|visitor center|historic place|historical landmark|sightseeing)\b'),
    'running': re.compile(r'\b(running|running track|athletics track|marathon|jogging)\b'),
    'water_sports': re.compile(r'\b(water sport|surf|scuba|snorkel|kayak|rafting|paddleboard|sailing|boat tour|marina)\b'),
    'walking': re.compile(r'\b(walking|walkway|walk of fame|promenade|boardwalk|pedestrian|garden|arboretum)\b'),
    'nightlife': re.compile(r'\b(nightlife|night club|nightclub|bar|lounge|casino|cocktail)\b'),
    'hiking': re.compile(r'\b(hiking|hiking area|trail|national park|mountain|peak|wilderness)\b'),
    'photography': re.compile(r'\b(photography|photo spot|viewpoint|scenic overlook|scenic spot|lookout)\b'),
    'zoo': re.compile(r'\b(zoo|aquarium|safari park|wildlife park|animal park)\b'),
    'shopping': re.compile(r'\b(shopping|shopping mall|shopping center|market|store|outlet|boutique|department store)\b'),
    'art gallary/museum': re.compile(r'\b(museum|art gallery|gallery|art museum|planetarium)\b'),
}

def categorize_place(row):
    fields = ('name', 'types', 'address', 'search_query')
    text = ' '.join(str(row.get(field, '') or '') for field in fields).casefold()
    text = re.sub(r'[_/]+', ' ', text)
    categories = [label for label, pattern in CATEGORY_RULES.items() if pattern.search(text)]

    # The query itself gives a useful category when the place name is vague.
    query = str(row.get('search_query', '') or '').casefold()
    if 'top tourist attractions' in query or 'historic sites and landmarks' in query:
        categories.append('sightseeing')
    if 'popular restaurants' in query:
        categories.append('restaurant_dining')
    if 'museums and cultural attractions' in query:
        categories.append('art gallary/museum')
    if 'parks, gardens, and outdoor attractions' in query:
        categories.extend(['walking', 'hiking', 'photography'])

    # These are tourist-place search results, so use sightseeing as a fallback.
    if not categories:
        categories = ['sightseeing']
    return ', '.join(dict.fromkeys(categories))

df['category'] = df.apply(categorize_place, axis=1)
df[['name', 'types', 'search_query', 'category']].head()


,name,types,search_query,category
0,Griffith Observatory,tourist_attraction;planetarium;museum;point_of...,"top tourist attractions in Los Angeles, Califo...","sightseeing, art gallary/museum"
1,Hollywood Walk of Fame,historical_landmark;tourist_attraction;histori...,"top tourist attractions in Los Angeles, Califo...","sightseeing, walking"
2,Hollywood Wax Museum,tourist_attraction;scenic_spot;museum;point_of...,"top tourist attractions in Los Angeles, Califo...","sightseeing, photography, art gallary/museum"
3,The Getty,art_museum;non_profit_organization;tourist_att...,"top tourist attractions in Los Angeles, Califo...","sightseeing, art gallary/museum"
4,Griffith Park,park;tourist_attraction;sports_activity_locati...,"top tourist attractions in Los Angeles, Califo...",sightseeing
